# Parcial 1 - Precio del Diésel en Colombia
**IIND-4414 · Ingeniería Financiera · 2024-20**

Notebook organizado para resolver los 5 puntos del parcial.
Funciones tomadas de los notebooks del curso (S5–S7) y del archivo base `Diesel parcial base`.

In [ ]:
# ── Imports ──
import numpy as np
import pandas as pd
import math

# Datos
import yfinance as yfin
from fredapi import Fred

# Gráficos
import matplotlib.pyplot as plt
from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm

# Estadística
from scipy.stats import norm, jarque_bera
from scipy import stats

# GARCH
from arch import arch_model

# Engle ARCH-LM
from statsmodels.stats.diagnostic import het_arch

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── FRED API ──
fred = Fred(api_key='a3d8a45391e92cc5c30c9c255819c138')


## 1. Carga de datos y cálculo del PMV

**Fuentes:**
- ULSD (DDFUELUSGULF): FRED
- TRM (COP=X): Yahoo Finance

**Fórmulas del enunciado:**
- $PPE_t = [PULSD_t - FL - CT] \times TRM_t$
- $PMV_t = 0.9 \cdot PPE_t + 0.1 \cdot PBD + Impuestos + Logística$


In [ ]:
# ── 1.1 Descarga de datos ──

# ULSD – Ultra Low Sulfur Diesel, U.S. Gulf Coast (USD/Gal)
df_d = pd.DataFrame(fred.get_series('DDFUELUSGULF', '2022-01-01', '2024-09-24'))
df_d = df_d.dropna()
df_d.index.names = ['Date']
df_d.columns = ['PULSD']
    
# TRM – Tasa Representativa del Mercado (COP/USD)
df_trm = yfin.download('COP=X', start='2022-01-01', end='2024-09-24', multi_level_index=False)
df_trm = df_trm[['Close']].rename(columns={'Close': 'TRM'})
df_trm.index.names = ['Date']
df_trm['TRM'] = df_trm['TRM'].ffill()

# Merge
df = pd.merge(df_d, df_trm, on='Date')
print(f"Observaciones: {len(df)}")
df


In [ ]:
# ── 1.2 Parámetros del enunciado ──
FL  = 0.12      # Fletes marítimos (USD/Gal)
CT  = 0.05      # Transporte por poliducto (USD/Gal)
PBD = 17_106    # Ingreso al productor biodiesel (COP/Gal)
IMP = 1_529     # Impuestos totales (COP/Gal)
LOG = 2_182     # Logística total (COP/Gal)


In [ ]:
# ── 1.3 Cálculo de PPE y PMV ──
df['PPE'] = (df['PULSD'] - FL - CT) * df['TRM']
df['PMV'] = 0.9 * df['PPE'] + 0.1 * PBD + IMP + LOG

# Precio spot actual (última observación)
S0 = df['PMV'].iloc[-1]
print(f"S₀ (PMV actual) = ${S0:,.2f} COP/Gal")
print(f"Fecha: {df.index[-1].strftime('%Y-%m-%d')}")


In [ ]:
# ── 1.4 Gráfico del PMV ──
plt.figure(figsize=(14, 5))
plt.plot(df['PMV'], color='darkblue', linewidth=0.8)
plt.title('Precio Máximo de Venta del Diésel (PMV)')
plt.ylabel('COP / Galón')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Punto 1 - Precio del Futuro ($F_0$)

**Escenario 3** (commodity con almacenamiento como tasa):

$$F_0 = S_0 \cdot e^{(r + u) \cdot T}$$

- $r_{COP}$ = 10.75% c.c.
- $u$ = 1.00% (almacenamiento)
- $T$ = 3/12


In [ ]:
# ── Punto 1: Precio del futuro ──
r_cop = 0.1075   # Tasa libre de riesgo COP (c.c.)
u     = 0.01     # Costo de almacenamiento
T     = 3 / 12   # Vencimiento: 3 meses

F0 = S0 * np.exp((r_cop + u) * T)
print(f"F₀ = ${F0:,.2f} COP/Gal")

## Punto 2 - Log-retornos y pruebas estadísticas

1. Calcular y graficar log-retornos
2. Jarque-Bera → ¿normalidad?
3. Ljung-Box sobre $r_t$ → ¿autocorrelación en media?
4. Ljung-Box sobre $r_t^2$ → ¿efecto ARCH?
5. Engle ARCH-LM → confirmar efecto ARCH


In [ ]:
# ── 2.1 Log-retornos ──
df['ret'] = np.log(df['PMV']) - np.log(df['PMV'].shift(1))
df['ret'].iloc[0] = 0
ret = df['ret']

# Retorno anualizado
ret_anual = np.mean(ret) * 252
print(f"Retorno anualizado (log): {ret_anual:.4%}")
print(f"Observaciones de retornos: {len(ret)}")


In [ ]:
# ── 2.2 Gráfico de log-retornos ──
plt.figure(figsize=(14, 5))
plt.plot(ret, color='red', linewidth=0.6)
plt.title('Log-Retornos del PMV del Diésel')
plt.ylabel('Retorno')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ── 2.3 Histograma ──
k = int(math.sqrt(len(ret)))
plt.figure(figsize=(10, 5))
plt.hist(ret, bins=k, edgecolor='black', linewidth=0.5)
plt.xlabel('Retorno')
plt.ylabel('Frecuencia')
plt.title('Histograma de Log-Retornos')
plt.tight_layout()
plt.show()


In [ ]:
# ── 2.4 Jarque-Bera ──
jb_stat, jb_p = stats.jarque_bera(ret)
skew = stats.skew(ret)
kurt = stats.kurtosis(ret)  # exceso de curtosis

print(f"Jarque-Bera = {jb_stat:.2f} | p-value = {jb_p:.2e}")
print(f"Asimetría (S) = {skew:.3f}")
print(f"Exceso de curtosis (C) = {kurt:.2f}")
print()
if jb_p < 0.05:
    print("→ RECHAZA H₀: los retornos NO son normales.")
else:
    print("→ NO se rechaza H₀: sin evidencia contra normalidad.")


In [ ]:
# ── 2.5 Ljung-Box sobre r_t (5 rezagos) ──
lags = 5
lb_r = sm.stats.diagnostic.acorr_ljungbox(ret, lags=lags)
Q_r = lb_r.iloc[-1, 0]
p_r = lb_r.iloc[-1, 1]

print(f"Ljung-Box r_t (5 lags): Q = {Q_r:.2f} | p = {p_r:.4f}")
if p_r < 0.05:
    print("→ RECHAZA H₀: hay autocorrelación en los retornos.")
else:
    print("→ NO se rechaza H₀: retornos independientes en media.")


In [ ]:
# ── 2.6 Ljung-Box sobre r_t² (5 rezagos) = prueba de efecto ARCH ──
ret2 = ret ** 2
lb_r2 = sm.stats.diagnostic.acorr_ljungbox(ret2, lags=lags)
Q_r2 = lb_r2.iloc[-1, 0]
p_r2 = lb_r2.iloc[-1, 1]

print(f"Ljung-Box r_t² (5 lags): Q = {Q_r2:.2f} | p = {p_r2:.2e}")
if p_r2 < 0.05:
    print("→ RECHAZA H₀: hay efecto ARCH (la varianza tiene memoria).")
else:
    print("→ NO se rechaza H₀: varianza constante.")


In [ ]:
# ── 2.7 Engle ARCH-LM (5 rezagos) ──
arch_test = het_arch(ret, nlags=5)
lm_stat = arch_test[0]
lm_p = arch_test[1]

print(f"Engle ARCH-LM (5 lags): LM = {lm_stat:.2f} | p = {lm_p:.2e}")
if lm_p < 0.05:
    print("→ RECHAZA H₀: confirma efecto ARCH.")
else:
    print("→ NO se rechaza H₀: sin efecto ARCH.")


In [ ]:
# ── 2.8 ACF de retornos y retornos² ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ret, lags=21, ax=axes[0], title='ACF de r_t')
plot_acf(ret2, lags=21, ax=axes[1], title='ACF de r_t²')
plt.tight_layout()
plt.show()


## Punto 3 - GARCH(1,1) y volatilidad anual

Se ajusta un GARCH(1,1) a los log-retornos y se evalúa con Ljung-Box sobre los residuales estandarizados al cuadrado (5 rezagos).


In [ ]:
# ── 3.1 Ajustar GARCH(1,1) ──
model = arch_model(ret, mean='Zero', vol='GARCH', p=1, q=1, rescale=False)
res = model.fit(disp='off')
print(res.summary())

In [ ]:
# ── 3.2 Parámetros estimados ──
omega = res.params['omega']
alpha = res.params['alpha[1]']
beta  = res.params['beta[1]']
persist = alpha + beta

print(f"ω = {omega:.6e}")
print(f"α = {alpha:.4f}")
print(f"β = {beta:.4f}")
print(f"Persistencia (α + β) = {persist:.4f}")
print(f"σ largo plazo = {np.sqrt(omega / (1 - persist)) * np.sqrt(252):.4%}")


In [ ]:
# ── 3.3 Ljung-Box sobre residuales² (5 rezagos) ──
std_resid = res.std_resid
lb_resid = sm.stats.diagnostic.acorr_ljungbox(std_resid ** 2, lags=5)
Q_res = lb_resid.iloc[-1, 0]
p_res = lb_resid.iloc[-1, 1]

print(f"Ljung-Box residuales² (5 lags): Q = {Q_res:.2f} | p-value = {p_res:.4f}")
if p_res >= 0.05:
    print("→ NO se rechaza H₀: el GARCH capturó la heterocedasticidad. Modelo adecuado.")
else:
    print("→ RECHAZA H₀: queda ARCH sin modelar. Limitación del modelo.")


In [ ]:
# ── 3.4 Volatilidad condicional y último valor ──
df['vol_garch'] = res.conditional_volatility * np.sqrt(252)
sigma_final = df['vol_garch'].iloc[-1]
print(f"Último valor de volatilidad anual (GARCH): σ = {sigma_final:.4%}")

plt.figure(figsize=(14, 5))
plt.plot(df['vol_garch'], color='darkred', linewidth=0.8)
plt.title('Volatilidad Anualizada - GARCH(1,1)')
plt.ylabel('σ anual')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Punto 4 - Collar Europeo (Black-Scholes-Merton)

Estrategia collar = **short put** (K=\$11,000) + **long call** (K=\$14,000), T = 3/12.

Prima neta = call_price − put_price  (la put vendida genera ingreso).

Se usa `black_scholes_f` del curso (commodity con almacenamiento).


In [ ]:
# ── Función BSM para commodities (del curso: S6_BSM_Oil) ──
def black_scholes_f(S, K, r, u, T, sigma, option):
    """
    Calculate the price of a European call or put option using the Black-Scholes model.
    Parameters:
        S (float): underlying asset price
        K (float): option strike price
        r (float): risk-free interest rate
        u (float): storage cost
        T (float): time to maturity in years
        sigma (float): volatility of underlying asset returns
        option (str): type of option to be priced, either 'call' (default) or 'put'
    Returns:
        price (float): price of the option
    """
    d1 = (np.log(S / K) + (r + u + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        price = np.exp(-r * T) * (S * np.exp((r + u) * T) * norm.cdf(d1) - K * norm.cdf(d2))
    elif option == 'put':
        price = np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp((r + u) * T) * norm.cdf(-d1))
    else:
        raise ValueError("Opción inválida: 'call' o 'put'")
    return price

In [ ]:
# ── 4.1 Valoración del collar ──
sigma = sigma_final  # última volatilidad del GARCH(1,1)
K_put  = 11000
K_call = 14000
u = 0.01
T = 3 / 12  # vencimiento: 3 meses

call_price = black_scholes_f(S0, K_call, r_cop, u, T, sigma, 'call')
put_price  = black_scholes_f(S0, K_put,  r_cop, u, T, sigma, 'put')

# Prima neta del collar: pagas el call, recibes la put
prima_collar = call_price - put_price

print(f"S₀ = ${S0:,.2f}")
print(f"σ  = {sigma:.4%}")
print(f"Call (K={K_call:,}): ${call_price:,.2f}")
print(f"Put  (K={K_put:,}):  ${put_price:,.2f}")
print(f"Prima neta del collar: ${prima_collar:,.2f}")


## Punto 5 - Collar Americano (Árbol Binomial, Δt = 1/12)

Misma estrategia pero con opciones que se pueden ejercer diariamente si están ITM.
Árbol binomial con pasos mensuales (Δt = 1/12), T = 3/12 → 3 pasos.

Parámetros del árbol (Sesión VIII):
- $u = e^{\sigma \sqrt{\Delta t}}$
- $d = e^{-\sigma \sqrt{\Delta t}}$
- $a = e^{(r+u_{stor}) \cdot \Delta t}$ (growth factor para commodity)
- $p = \frac{a - d}{u - d}$


In [ ]:
# ── Función: árbol binomial para opciones americanas (commodity) ──
def binomial_american(S, K, r, u_stor, T, sigma, N, option):
    """
    Árbol binomial para opción americana sobre commodity.
    
    Parámetros:
        S       : precio spot
        K       : strike
        r       : tasa libre de riesgo c.c.
        u_stor  : costo de almacenamiento (tasa)
        T       : tiempo a vencimiento (años)
        sigma   : volatilidad anual
        N       : número de pasos
        option  : 'call' o 'put'
    Retorna:
        precio de la opción
    """
    dt = T / N
    up = np.exp(sigma * np.sqrt(dt))
    dn = 1 / up
    a  = np.exp((r + u_stor) * dt)  # growth factor (commodity)
    p  = (a - dn) / (up - dn)
    disc = np.exp(-r * dt)

    # Precios del activo en cada nodo final
    ST = np.array([S * up**j * dn**(N - j) for j in range(N + 1)])

    # Payoff en nodos finales
    if option == 'call':
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)

    # Backward induction con ejercicio temprano
    for i in range(N - 1, -1, -1):
        Si = np.array([S * up**j * dn**(i - j) for j in range(i + 1)])
        V_hold = disc * (p * V[1:i+2] + (1 - p) * V[0:i+1])
        if option == 'call':
            V_exercise = np.maximum(Si - K, 0)
        else:
            V_exercise = np.maximum(K - Si, 0)
        V = np.maximum(V_hold, V_exercise)

    return V[0]


In [ ]:
# ── 5.1 Valoración del collar con árbol binomial ──
N = 3  # pasos mensuales (T=3/12, dt=1/12)

call_am = binomial_american(S0, K_call, r_cop, u, T, sigma, N, 'call')
put_am  = binomial_american(S0, K_put,  r_cop, u, T, sigma, N, 'put')

prima_collar_am = call_am - put_am

print(f"Call americana (K={K_call:,}): ${call_am:,.2f}")
print(f"Put  americana (K={K_put:,}):  ${put_am:,.2f}")
print(f"Prima neta del collar (binomial): ${prima_collar_am:,.2f}")
